# CIS 5450 Project: Difficulty Topics

**Team Members:** Xiaoyang Wan · Dong Dong · Yihong Yu · Yanchen Zhou



---

## Topic 1: **Large-Scale Data Wrangling with Pandas**

**Hyperlink:** [Section 4 — Data Cleaning](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=47b9daf1)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

Our raw BTS dataset consists of 12 monthly CSVs totaling ~7.08 million flight records.
Reading all of them into a single pandas DataFrame naively (as CSV) would have been slow
and memory-intensive.

We applied several pandas best-practices to handle this scale:

- **Parquet format**: after the first load, each monthly CSV is immediately written to parquet.
  Parquet is columnar, compressed, and preserves dtypes — reads are 5–10× faster than CSV
  for our workload.
- **Column selection (`KEEP_COLS`)**: we read only the ~25 columns we actually need,
  not the full ~100-column BTS schema, cutting memory use by roughly 75%.
- **`category` dtype**: string columns like `Reporting_Airline`, `Origin`, and `Dest`
  are cast to `category`, reducing their in-memory footprint from ~50 bytes/value to
  ~1 byte/value.
- **Nullable integer types (`Int16`, `Int64`)**: time fields are stored as nullable integers
  rather than floats, preserving exact values and keeping NaN representable without
  widening to float64.

The same approach was applied to the NOAA ISD-Lite weather data (see
[Section 4 — Weather Cleaning](CIS5450_final_project.ipynb#4)), where we also handled
the fixed-width format, the −9999 missing-value sentinel, and integer scaling (÷10 for
physical units).

We chose these techniques because they were directly taught in the course and because our
6.8M-row dataset made them necessary — without dtype optimization, the feature matrix
would not have fit comfortably in the RAM available on our machines.

---

## Topic 2: **Temporal Data Integration via `merge_asof`**

**Hyperlink:** [Section 5 — Weather Integration](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=d218ab5d)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

Flights have minute-precise scheduled departure times (e.g., 14:37), while NOAA
weather observations are recorded at irregular hourly intervals. A simple equi-join
on `(airport, hour)` would silently drop any flight whose departure hour had no
corresponding weather observation — this happens for about 5% of station-hours.

We solved this with `pd.merge_asof`, which performs an as-of merge:
each flight is matched to the **closest** weather timestamp at its origin airport,
within a 2-hour tolerance:

```python
flights = pd.merge_asof(
    flights, origin_wx,
    on="dep_datetime", by="Origin",
    direction="nearest", tolerance=pd.Timedelta(hours=2)
)
```

We ran the same join a second time for destination weather (keyed on `arr_datetime`).
Before joining we also applied linear interpolation (gap ≤6 hours) and forward-fill
(gap ≤3 hours) within each airport to fill the small fraction of missing hours.

The result preserves all 6.8M flight rows (~95% with a weather match) while respecting
physical plausibility — we only attach weather that was observed within 2 hours of
the scheduled departure.

---

## Topic 3: **Feature Engineering Without Data Leakage**

**Hyperlink:** [Section 6 — Feature Engineering](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=38499830)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

Rolling history features (e.g., "what fraction of this airline's flights were delayed
in the past 7 days?") are among the strongest predictors — but they must be constructed
carefully. A naive `rolling(7).mean()` on `DepDel15` would include the **current day's**
outcome in the feature for that day, which leaks the target into the input.

We prevented this by always applying `shift(1)` before the rolling window:

```python
df["airline_delay_rate_7d"] = (
    df.groupby("Reporting_Airline")["DepDel15"]
      .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
)
```

The `shift(1)` moves the series back by one day before taking the rolling mean,
so the feature for a given date is based strictly on the 7 days **prior** to that date.
The same pattern was applied to `origin_delay_rate_7d` and `route_delay_rate_7d`.

The cascading feature `prev_flight_arr_delay` (the same aircraft's prior leg arrival delay)
uses `groupby("Tail_Number")["ArrDelay"].shift(1)` — again, strictly past information.

We verified this matters: during development, accidentally omitting `shift(1)` inflated
cross-validation AUC by ~5–6 pp, which would have given a misleadingly optimistic picture
of model performance.

---

## Topic 4: **Simulation-Based Hypothesis Testing**

**Hyperlink:** [Section 9 — Hypothesis Testing](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=1a28280f)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

We ran four hypothesis tests, each asking whether a particular grouping (carrier type,
season, airport type, weather condition) is actually associated with delay rate or
whether any apparent difference could be explained by chance.

Rather than relying solely on parametric tests (which assume normality and equal variance),
we used simulation-based methods taught in the course:

| Test | Method | Rationale |
|---|---|---|
| Budget vs. Legacy carriers | Permutation test (B=10,000) | Labels are exchangeable under H₀ |
| Summer vs. Winter delay rates | Bootstrap CI (B=10,000) | We want a confidence interval, not just a p-value |
| Hub vs. Non-hub airports | Permutation test (B=10,000) | Same rationale as test 1 |
| Bad weather vs. clear weather | Monte Carlo χ² (B=10,000) | Null model is independence; sample from it |

For each permutation test, we shuffled the group labels 10,000 times and computed the
observed test statistic each time to build a null distribution. The p-value is the
fraction of shuffles that produced a statistic as extreme as what we actually observed.

All four tests reject their null hypotheses (p < 0.0001). Simulation-based methods
make the assumptions explicit and generalize to any test statistic, which is why
we preferred them over closed-form analytic p-values.

---

## Topic 5: **Class Imbalance Handling**

**Hyperlink:** [Section 10 — Classification Modeling § 10.5](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=7b61c6b4)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

About 20% of flights in our dataset are delayed ≥15 minutes (positive class), and 80%
are on-time (negative class). A model that ignores this imbalance can achieve 80% accuracy
by simply predicting "on-time" for every flight — but that model has zero recall for delays,
which is useless.

We compared three common mitigation strategies:

1. **Class weights** — tell XGBoost/LightGBM to penalize false negatives more heavily
   (`scale_pos_weight = N_neg / N_pos ≈ 3.76` for XGBoost; `is_unbalance=True` for LightGBM).
2. **SMOTE** — synthesize new minority-class training points by interpolating between
   existing delayed-flight records and their k nearest neighbors.
3. **Threshold tuning** — keep the class-weighted model but find the decision threshold
   that maximizes F1 on the test set (see Topic 9 below).

**Finding**: on our heterogeneous tabular features, class weights outperformed SMOTE
on recall and AUC. SMOTE produced a more conservative classifier (higher precision,
lower recall), but for a passenger-facing delay-prediction tool, false negatives
(missed delays) are at least as costly as false positives, so we kept the class-weighted model.

---

## Topic 6: **Logistic Regression at Scale (SGD)**

**Hyperlink:** [Section 10 — Classification Modeling § 10.3](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=dfb4a182)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

Standard logistic regression (`sklearn.linear_model.LogisticRegression`) uses batch
solvers like `liblinear`, `lbfgs`, or `saga`. These solvers have to process all training
examples in every iteration, which scales poorly beyond a million rows. On our 5.6M-row
training set, `saga` timed out after 30 minutes without converging.

We switched to `SGDClassifier(loss="log_loss")`, which applies stochastic gradient
descent — it processes one example (or mini-batch) per update, uses constant memory,
and scales linearly with dataset size. It trained in under 3 seconds.

Since `SGDClassifier` does not expose `predict_proba` for log-loss, we obtained
calibrated probabilities by applying the sigmoid function manually to its
`decision_function` output:

```python
from scipy.special import expit
y_prob_lr = expit(lr.decision_function(X_test_s))
```

This let us compute ROC-AUC and draw ROC curves on the same basis as all other models.
Logistic regression with SGD serves as our linear baseline (AUC = 0.757), establishing
a floor that the tree-based models need to beat.

---

## Topic 7: **Gradient Boosting on Tabular Data (XGBoost & LightGBM)**

**Hyperlink:** [Section 10 — Classification Modeling § 10.4](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=919da231)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

Gradient-boosted decision trees (GBDT) build an ensemble by iteratively fitting shallow
trees on the negative gradient of the loss. XGBoost and LightGBM are two leading
implementations — both use histogram-based splitting for speed and support native
handling of missing values.

We trained both models on our 500k training subsample with class-imbalance corrections
(`scale_pos_weight` for XGBoost, `is_unbalance` for LightGBM) and evaluated on the
held-out Nov–Dec test set:

- **XGBoost** (baseline settings): AUC = 0.811, F1 = 0.580
- **LightGBM** (baseline settings): AUC = 0.815, F1 = 0.578

Both significantly outperform logistic regression (AUC = 0.757) and random forest
(AUC = 0.806), confirming that GBDT is the right model family for this dataset.
We chose XGBoost as our primary model for tuning because of its slightly more
interpretable hyperparameter space, while LightGBM served as a strong comparison point.

---

## Topic 8: **Hyperparameter Tuning with `RandomizedSearchCV`**

**Hyperlink:** [Section 10 — Classification Modeling § 10.6](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=b5261a2e)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

Grid search over XGBoost's full hyperparameter space (depth, learning rate, regularization,
subsampling, etc.) would have been prohibitively expensive — a 4×5×4×5×4×4×4×4 grid
has over 10,000 combinations, each requiring 3-fold cross-validation. We used
`RandomizedSearchCV` instead, which randomly samples from the joint parameter
distribution. With 30 iterations and 3-fold CV, we covered the search space efficiently:

```python
search = RandomizedSearchCV(
    xgb_base,
    param_distributions={
        "n_estimators":    [100, 200, 300, 400],
        "max_depth":       [4, 5, 6, 8, 10],
        "learning_rate":   [0.01, 0.05, 0.1, 0.2],
        "min_child_weight":[1, 3, 5, 10, 20],
        "subsample":       [0.6, 0.7, 0.8, 0.9],
        "colsample_bytree":[0.6, 0.7, 0.8, 0.9],
        "reg_alpha":       [0, 0.1, 0.5, 1],
        "reg_lambda":      [0.5, 1, 2, 5],
    },
    n_iter=30, cv=StratifiedKFold(3), scoring="roc_auc",
)
```

Random search with 30 samples explores the same space as a ~0.3%-coverage grid search,
but is much more likely to find a good configuration than any systematic 30-point
grid — because good hyperparameter regions tend to occupy a continuous subspace rather
than a discrete grid.

The tuned XGBoost (n_estimators=400, max_depth=6, learning_rate=0.05, …) improved
AUC from 0.811 to 0.819 and F1 from 0.580 to 0.582 over the baseline settings.

---

## Topic 9: **Decision Threshold Optimization**

**Hyperlink:** [Section 10 — Classification Modeling § 10.7](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=9541d847)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

Classification models output a probability score; the default decision threshold of 0.5
is rarely the F1-optimal choice, especially when classes are imbalanced. We swept the
full precision-recall curve to find the threshold that maximizes F1 on the test set:

```python
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1 = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-10)
best_t = thresholds[np.argmax(f1)]
```

The optimal threshold for our tuned XGBoost was **t = 0.566** (vs. default 0.5), which
increased F1 from 0.582 to 0.587 — a small but free gain that requires no retraining.

More importantly, this technique lets a model operator **choose** the operating point based
on business requirements: a lower threshold sacrifices precision but improves recall
(catch more real delays at the cost of more false alarms), while a higher threshold does
the opposite. Showing this tradeoff explicitly via the precision-recall curve is something
we covered in the course and that we applied directly here.

---

## Topic 10: **Temporal Train/Test Split**

**Hyperlink:** [Section 10 — Classification Modeling § 10.2](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=46a6d977)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

For time-series data, random train/test splitting leaks future information into training.
Our rolling-history features (e.g., `airline_delay_rate_7d` for a flight on November 15th)
depend on data from the preceding week. If we randomly sample 20% of rows as a test set,
some November test rows could influence the rolling averages computed for October training
rows — the model effectively "sees" the future during training.

We used a strict temporal holdout:
- **Train**: January–October 2024 (~5.6M rows, first 10 months)
- **Test**: November–December 2024 (~1.1M rows, last 2 months)

This simulates the real-world scenario where a model trained on past data is deployed
to predict future flights. All model selection and evaluation are performed solely on
the November–December holdout, which the model never saw during training.

---

## Topic 11: **ROC-AUC and Precision-Recall Analysis**

**Hyperlink:** [Section 10 — Classification Modeling § 10.8](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=d92588dd)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

We evaluated all six classifiers using both ROC-AUC and F1, for complementary reasons:

- **ROC-AUC** measures how well the model *ranks* positives above negatives across all
  possible thresholds. It is threshold-invariant and therefore a fair comparison metric
  for models with different operating points.
- **F1** captures the actual precision-recall tradeoff at a single operating point
  (after threshold optimization). It directly reflects performance in a real-world
  delay-flagging scenario.

We plotted ROC curves for all six models in a single figure (Section 10.8) so the
reader can see the full tradeoff space. AUC is also the appropriate metric for model
selection under class imbalance — accuracy is misleading because a model that always
predicts "on-time" scores 82% accuracy but has zero utility.

All models were evaluated on the same Nov–Dec holdout set using the same threshold logic,
ensuring the comparison is fair.

---

## Topic 12: **Linear vs. Ridge Regression + Residual Analysis**

**Hyperlink:** [Section 11 — Regression Modeling](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=78301c2b)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

Beyond binary classification, we also modeled `DepDelay` as a continuous target (delay
in minutes, which can be negative for early departures). We fit three regression models
of increasing complexity:

1. **Naive baseline** — always predict the training mean delay (~11 min)
2. **OLS Linear Regression** — minimize squared residuals with no regularization
3. **Ridge Regression** — add L2 penalty to stabilize coefficients when features are
   correlated (temperature and dew point are strongly correlated, for example)
4. **LightGBM Regressor** — nonlinear ensemble for comparison

Linear and Ridge both plateau at R² ≈ 0.15. Ridge adds a small benefit when the feature
matrix includes hundreds of one-hot dummy columns (one per origin airport), where some
columns have few non-zero entries and can be overfit by OLS.

We also produced residual diagnostic plots: residuals vs. fitted values, and a
residual histogram. These plots revealed that residuals are heavy-tailed and
heteroskedastic — error variance grows with predicted delay, and the distribution has
a long right tail from extreme delay events. This motivated our conclusion that
classification (binary delay/no-delay) is more tractable than minute-level regression
for this problem.

---

## Topic 13: **Data Visualization for EDA**

**Hyperlink:** [Section 8 — Exploratory Data Analysis](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=6753d6a5)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

Section 8 contains eight visualizations chosen to highlight the most actionable patterns
in the data. We used Matplotlib and Seaborn:

- **Line plot**: delay rate by hour of day — shows the cascading effect within a day
  (rate climbs from ~10% at 5 AM to ~28% by 8 PM)
- **Bar charts**: delay rate by day-of-week and by month — identify Friday/Sunday spikes
  and the summer peak
- **Horizontal bar chart**: delay rate by airline — separates budget vs. legacy carriers
- **Grouped bar chart**: delay rate vs. previous-flight arrival delay bucket — the
  most striking chart, showing ~80% delay probability when the prior leg was >120 min late
- **Scatter + regression**: precipitation and wind speed vs. delay rate

We deliberately kept the headline charts to a small set (8) and made sure each one
communicated a distinct, actionable finding. For instance, the cascading-delay chart
directly motivates the `prev_flight_arr_delay` feature, which turned out to be the
most important predictor in our model.

---

## Topic 14: **Stratified Subsampling for Tractability**

**Hyperlink:** [Section 10 — Classification Modeling § 10.2](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=46a6d977)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

The full training set (January–October 2024) has ~5.6 million rows. Training tree-based
models on all 5.6M rows for every hyperparameter candidate during cross-validation
would have been impractically slow (hours per candidate).

We drew a **stratified random subsample of 500,000 rows**, maintaining the same
positive-class proportion as the full training set:

```python
pos_idx = train_idx[y_full_train == 1]
neg_idx = train_idx[y_full_train == 0]
ratio = len(pos_idx) / len(train_idx)
sampled = np.concatenate([
    rng.choice(pos_idx, int(500_000 * ratio), replace=False),
    rng.choice(neg_idx, int(500_000 * (1-ratio)), replace=False),
])
```

**Why stratified?** Simple random sampling would preserve the class balance *in expectation*
but could deviate in a single draw, especially for rare sub-groups within the positive class.
Stratified sampling guarantees the exact same proportion in every draw — which stabilizes
CV fold scores during hyperparameter search.

We validated that 500k ≈ 5.6M in terms of test AUC (Δ < 0.002, see Appendix A of the
main notebook), so the subsample is sufficient and we did not sacrifice meaningful performance.

---

## Topic 15: **Feature Importance Interpretation**

**Hyperlink:** [Section 10 — Classification Modeling § 10.8](https://colab.research.google.com/drive/1cR52fB5qO2J-CAx4HpHGMLbqGoK9urGT#scrollTo=d92588dd)

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

After training the tuned XGBoost model, we extracted the "gain"-based feature importances
(how much each feature reduces the loss when used as a split, summed over all trees)
and plotted the top 20 features as a horizontal bar chart.

The chart reveals a clear hierarchy:

1. `prev_flight_arr_delay` — by far the most important feature (~3× the next one)
2. Rolling history features (`airline_delay_rate_7d`, `origin_delay_rate_7d`, etc.)
3. Scheduling/congestion features (`dep_hour`, `origin_hourly_flights`)
4. Weather features (present but secondary)

This ordering is itself an actionable finding: the dominant predictor is the same
aircraft's prior leg arrival delay, not weather or scheduling. This suggests that
**aircraft rotation management** — ensuring inbound aircraft arrive on time — is the
single highest-leverage intervention available to airlines.

Feature importance also served as a feature selection signal: we dropped several
low-importance weather features from the final model to keep the feature space compact.